> **หมายเหตุ:** outputs ที่บันทึกไว้อาจเก่ากว่าโค้ดปัจจุบัน; รันเซลล์ inference ใหม่ก่อนใช้ผล และหน้าต่างสุดท้ายจะเลื่อนให้จบตรงท้ายไฟล์; ไฟล์ที่สั้นกว่า 4 วินาทีจะถูกระบุว่า limited_context

# RakForest — ทดลองโมเดล multi-label กับไฟล์เสียงใหม่

ใช้ checkpoint รุ่น `multilabel-v1` แบ่งเสียงเป็นช่วงละ 4 วินาที และรายงานได้หลายคลาสพร้อมกันในแต่ละช่วง. ผลรายช่วงและสรุปไฟล์บันทึกไว้ใน `reports/`.

## ข้อควรรู้ก่อนใช้

Multi-label model ให้ score แยกแต่ละคลาส; หลายคลาสอาจผ่าน threshold พร้อมกัน หรือไม่มีคลาสใดผ่านก็ได้. ค่า threshold 0.5/0.75 เป็นค่าเริ่มต้นเพื่อทดลอง ยังไม่ calibrate กับเสียงภาคสนาม. คะแนนไม่ยืนยันเหตุ และโมเดลยังไม่รู้จักเสียงนอกคลาสโดยอัตโนมัติ.

## วิธีทำงานของไฟล์นี้: ทดลองทำนายเสียงใหม่

ไฟล์นี้เป็นขั้น **inference/ทดลองใช้งาน checkpoint** ไม่ได้ฝึกโมเดล ใช้ checkpoint multi-label จาก 03, เตรียมเสียงใหม่ตาม sample rate/ความยาว/features เดียวกับตอนฝึก, แบ่งเป็นช่วง แล้วคำนวณ sigmoid score แยกคลาสในแต่ละช่วง; หน้าต่างสุดท้ายจะเลื่อนให้จบตรงท้ายไฟล์เพื่อให้มีเสียงจริงครบ 4 วินาที; เฉพาะไฟล์ทั้งไฟล์ที่สั้นกว่า 4 วินาทีจึงเติมความยาวและติดสถานะ `limited_context`

ผลหนึ่งช่วงจึงมีได้หลายคลาสที่คะแนนเกิน threshold พร้อมกัน ตารางผลเก็บคะแนนทุกคลาสและช่วงเวลาไว้เพื่อให้ตรวจทานได้ คะแนนและ threshold ปัจจุบันเป็นค่าทดลอง ยังไม่ใช่การแจ้งเหตุอัตโนมัติที่ผ่านการ calibrate

อัปโหลดหรือเลือกไฟล์เสียงได้จาก widget ใน notebook; ฟังผลและตรวจ false alarm/เสียงเงียบด้วยคนก่อนนำไปใช้จริง การทำนายบน notebook นี้ยังไม่เท่ากับการเชื่อม inference API เข้ากับเว็บไซต์


## 1. ติดตั้งเฉพาะ dependencies ที่ยังไม่มี

In [3]:
import importlib.util, subprocess, sys
required_modules={'torch':'torch','librosa':'librosa','soundfile':'soundfile','numpy':'numpy','pandas':'pandas','matplotlib':'matplotlib','IPython':'ipython','tqdm':'tqdm','ipywidgets':'ipywidgets'}
missing=[package for module,package in required_modules.items() if importlib.util.find_spec(module) is None]
if missing:
    print('Installing missing packages:',missing)
    subprocess.check_call([sys.executable,'-m','pip','install',*missing])
else:
    print('All required packages are already installed.')

All required packages are already installed.


## 2. Imports, paths และค่าที่ปรับได้

In [4]:
from pathlib import Path
import json, math
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch
from torch import nn
from IPython.display import Audio, display, Markdown
from tqdm import tqdm

PROJECT_ROOT=Path.cwd()
if PROJECT_ROOT.name=='notebooks': PROJECT_ROOT=PROJECT_ROOT.parent
MODEL_PATH=PROJECT_ROOT/'models'/'rakforest_audio_cnn_multilabel_best.pt'
INPUT_DIR=PROJECT_ROOT/'data'/'incoming_audio'
REPORT_DIR=PROJECT_ROOT/'reports'
INPUT_DIR.mkdir(parents=True,exist_ok=True); REPORT_DIR.mkdir(parents=True,exist_ok=True)
AUDIO_EXTENSIONS={'.wav','.mp3','.flac','.ogg','.m4a','.aac','.aiff','.aif'}
BATCH_SIZE=32
RECURSIVE_INPUT=True
REVIEW_CLASSES={'fire','chainsaw','human_activity','threat_tactical','vehicle'}
REVIEW_THRESHOLD=0.50  # provisional; calibrate on validation/field recordings later
ALERT_CLASSES={'fire','chainsaw','human_activity','threat_tactical','vehicle'}
ALERT_THRESHOLD=0.75  # provisional; not a calibrated alarm threshold
HAS_MPS=hasattr(torch.backends,'mps') and torch.backends.mps.is_available()
DEVICE=torch.device('cuda' if torch.cuda.is_available() else ('mps' if HAS_MPS else 'cpu'))
print('Project:',PROJECT_ROOT)
print('Input folder:',INPUT_DIR)
print('Device:',DEVICE)
print('Inference uses the model window; the final full window aligns to the file end when needed.')

Project: /Users/guidegiegg/Projects/RakForest-ML
Input folder: /Users/guidegiegg/Projects/RakForest-ML/data/incoming_audio
Device: mps
Inference uses the model window; the final full window aligns to the file end when needed.


## 3. โหลด checkpoint และสร้างสถาปัตยกรรมเดียวกับไฟล์ 03

In [5]:
if not MODEL_PATH.exists():
    raise FileNotFoundError(f'ไม่พบ {MODEL_PATH}; รัน 03_train_model.ipynb เพื่อฝึกโมเดลก่อน')
try:
    checkpoint=torch.load(MODEL_PATH,map_location=DEVICE,weights_only=True)
except TypeError:
    checkpoint=torch.load(MODEL_PATH,map_location=DEVICE)
if checkpoint.get('model_type')!='multilabel_audio_cnn' or checkpoint.get('schema_version')!='multilabel-v1':
    raise RuntimeError('Checkpoint นี้ยังเป็น single-label/รุ่นเก่า; รัน 03_train_model.ipynb รุ่น multi-label ให้เสร็จก่อน')
if not checkpoint.get('manifest_signature') or not checkpoint.get('training_datasets'):
    raise RuntimeError('Checkpoint เก่าขาด provenance ของชุดฝึก; ใช้ checkpoint ที่สร้างจาก 03_train_model.ipynb เวอร์ชันล่าสุด')
if 'zenodo_fire' in checkpoint['training_datasets']:
    raise RuntimeError('Checkpoint ฝึกด้วย Zenodo Fire ที่ถูกพักไว้; สร้าง manifest ใหม่และฝึกใหม่ก่อน inference')
class_order=list(checkpoint['class_names'])
config=checkpoint['config']
if not class_order or len(class_order)!=len(set(class_order)):
    raise ValueError('Checkpoint class_names ว่างหรือมีชื่อคลาสซ้ำ')
if config.get('architecture')!='AudioCNN-v2-multilabel' or config.get('output_activation')!='sigmoid':
    raise ValueError(f"สถาปัตยกรรมใน checkpoint ไม่รองรับ: {config.get('architecture')}")
required_config={'sample_rate','duration_seconds','n_mels','n_fft','hop_length'}
missing_config=required_config-set(config)
if missing_config: raise ValueError(f'Checkpoint config ขาดค่า: {sorted(missing_config)}')
class_to_index={name:i for i,name in enumerate(class_order)}
if checkpoint.get('class_to_index',class_to_index)!=class_to_index:
    raise ValueError('class_to_index ใน checkpoint ไม่ตรงกับ class_names')
SAMPLE_RATE=int(config['sample_rate']); DURATION_SECONDS=float(config['duration_seconds'])
NUM_SAMPLES=int(SAMPLE_RATE*DURATION_SECONDS)
N_MELS=int(config['n_mels']); N_FFT=int(config['n_fft']); HOP_LENGTH=int(config['hop_length'])

class ConvBlock(nn.Module):
    def __init__(self,in_channels,out_channels):
        super().__init__()
        self.block=nn.Sequential(nn.Conv2d(in_channels,out_channels,3,padding=1,bias=False),nn.BatchNorm2d(out_channels),nn.ReLU(inplace=True),nn.MaxPool2d(2))
    def forward(self,x): return self.block(x)

class AudioCNN(nn.Module):
    def __init__(self,num_classes):
        super().__init__()
        self.features=nn.Sequential(ConvBlock(1,16),ConvBlock(16,32),ConvBlock(32,64),ConvBlock(64,128),nn.AdaptiveAvgPool2d((1,1)))
        self.classifier=nn.Sequential(nn.Flatten(),nn.Dropout(.35),nn.Linear(128,num_classes))
    def forward(self,x): return self.classifier(self.features(x))

model=AudioCNN(len(class_order)).to(DEVICE)
model.load_state_dict(checkpoint['model_state_dict']); model.eval()
print('Checkpoint epoch:',checkpoint.get('best_epoch'))
print('Validation macro-F1 (multi-label):',checkpoint.get('best_val_macro_f1'))
print('Schema:',checkpoint['schema_version'],'| output: independent sigmoid scores')
print('Classes:',class_order)
print('Training datasets:',checkpoint['training_datasets'])
print('Training manifest signature:',checkpoint['manifest_signature'][:12])

Checkpoint epoch: 11
Validation macro-F1 (multi-label): 0.7138725486031142
Schema: multilabel-v1 | output: independent sigmoid scores
Classes: ['fire', 'chainsaw', 'vehicle', 'human_activity', 'threat_tactical', 'weather', 'natural_ambient', 'silence', 'other']
Training datasets: ['datasec_v2', 'esc50', 'fire_google_drive', 'footstep_set', 'fsc22', 'rfcx_frugalai', 'vehicle_sounds']
Training manifest signature: 21384c97ca16


In [6]:
import ipywidgets as widgets
from IPython.display import display

uploader=widgets.FileUpload(
    accept=','.join(sorted(AUDIO_EXTENSIONS)),
    multiple=True,
    description='เลือกไฟล์เสียง'
)
display(uploader)
print('เลือกไฟล์เสียงด้านบน แล้วรันเซลล์ถัดไปเพื่อบันทึก')

FileUpload(value=(), accept='.aac,.aif,.aiff,.flac,.m4a,.mp3,.ogg,.wav', description='เลือกไฟล์เสียง', multipl…

เลือกไฟล์เสียงด้านบน แล้วรันเซลล์ถัดไปเพื่อบันทึก


In [7]:
import shutil

def unique_destination(folder,name):
    candidate=folder/Path(name).name
    if not candidate.exists(): return candidate
    stem,suffix=candidate.stem,candidate.suffix
    counter=1
    while True:
        candidate=folder/f'{stem}_{counter}{suffix}'
        if not candidate.exists(): return candidate
        counter+=1

def uploaded_items(value):
    # ipywidgets 7 uses a dict; version 8 uses a tuple of file metadata dicts.
    if isinstance(value,dict):
        for name,metadata in value.items():
            yield name,metadata.get('content',b'')
    else:
        for metadata in value:
            yield metadata.get('name','audio'),metadata.get('content',b'')

uploaded=list(uploaded_items(uploader.value))
if not uploaded:
    print('ยังไม่มีไฟล์อัปโหลด; วางไฟล์ไว้ใน incoming_audio ด้วยตนเองได้')
else:
    saved=[]
    for name,content in uploaded:
        safe_name=Path(name).name
        if Path(safe_name).suffix.lower() not in AUDIO_EXTENSIONS:
            print('ข้ามไฟล์นามสกุลที่ไม่รองรับ:',safe_name); continue
        destination=unique_destination(INPUT_DIR,safe_name)
        destination.write_bytes(bytes(content))
        saved.append(destination)
    print(f'บันทึก {len(saved)} ไฟล์ลง {INPUT_DIR}')
    for path in saved: print(' -',path.name)

ยังไม่มีไฟล์อัปโหลด; วางไฟล์ไว้ใน incoming_audio ด้วยตนเองได้


## 4. อัปโหลดไฟล์เสียงเพื่อทำนาย

รันเซลล์ถัดไปเพื่อแสดงปุ่มเลือกไฟล์เสียง แล้วเลือกไฟล์จากเครื่องได้หลายไฟล์ เมื่อเลือกเสร็จให้รันเซลล์บันทึกไฟล์ ไฟล์จะถูกเก็บไว้ใน `data/incoming_audio/` และเซลล์ถัดไปจะพบไฟล์เหล่านั้นสำหรับ inference. หากเพิ่งติดตั้ง `ipywidgets` แต่ปุ่มไม่แสดง ให้ restart kernel แล้วรันเซลล์ตามลำดับอีกครั้ง.

In [8]:
searcher=INPUT_DIR.rglob('*') if RECURSIVE_INPUT else INPUT_DIR.glob('*')
audio_paths=sorted(p for p in searcher if p.is_file() and p.suffix.lower() in AUDIO_EXTENSIONS and not p.name.startswith('.'))
print(f'Found {len(audio_paths):,} audio files')
if not audio_paths:
    print(f'ยังไม่พบไฟล์เสียง ให้วางไฟล์ใน {INPUT_DIR} แล้วรัน cell นี้ใหม่')
else:
    display(pd.DataFrame({'file':[str(p.relative_to(PROJECT_ROOT)) for p in audio_paths]}).head(30))

Found 2 audio files


,file
0,data/incoming_audio/Car Passing By - Sound Eff...
1,data/incoming_audio/City Rain at Night Night T...


## 5. Preprocessing ให้ตรงกับ 02 และ feature extraction ให้ตรงกับ 03

In [9]:
def read_audio_mono(path):
    try:
        y,sr=sf.read(path,dtype='float32',always_2d=False)
        if y.ndim==2: y=y.mean(axis=1)
    except Exception:
        # Fallback decoder for formats such as MP3/M4A when libsndfile lacks codec support.
        y,sr=librosa.load(path,sr=None,mono=True,dtype=np.float32)
    return np.asarray(y,dtype=np.float32),int(sr)

def load_and_segment(path):
    y,sr=read_audio_mono(path)
    if y.size==0: return [],int(sr)
    y-=float(np.mean(y))  # DC offset removal, same as 02
    if sr!=SAMPLE_RATE: y=librosa.resample(y,orig_sr=sr,target_sr=SAMPLE_RATE)
    if len(y)<=NUM_SAMPLES:
        valid_samples=len(y)
        segment=np.pad(y,(0,NUM_SAMPLES-valid_samples)) if valid_samples<NUM_SAMPLES else y
        return [(segment.astype(np.float32),0.0,valid_samples/SAMPLE_RATE)],int(sr)
    # Keep the normal 4-second blocks, then align one final full window to the end
    # instead of classifying a tiny padded tail on its own.
    starts=list(range(0,len(y)-NUM_SAMPLES+1,NUM_SAMPLES))
    final_start=len(y)-NUM_SAMPLES
    if starts[-1]!=final_start: starts.append(final_start)
    segments=[]
    for start in starts:
        segment=y[start:start+NUM_SAMPLES]
        segments.append((segment.astype(np.float32),start/SAMPLE_RATE,DURATION_SECONDS))
    return segments,int(sr)

def to_logmel(y):
    mel=librosa.feature.melspectrogram(y=y,sr=SAMPLE_RATE,n_fft=N_FFT,hop_length=HOP_LENGTH,n_mels=N_MELS,power=2.0)
    logmel=librosa.power_to_db(mel,ref=np.max)
    return ((logmel-logmel.mean())/(logmel.std()+1e-6)).astype(np.float32)

print(f'Input standardization: mono, {SAMPLE_RATE} Hz, {DURATION_SECONDS:g} seconds per segment')

Input standardization: mono, 16000 Hz, 4 seconds per segment


## 6. รันทำนายและบันทึกผล

In [10]:
rows=[]; failures=[]
for path in tqdm(audio_paths,desc='Predict audio files'):
    try:
        segments,original_sr=load_and_segment(path)
        if not segments:
            failures.append({'file':str(path),'error':'empty audio'}); continue
        for batch_start in range(0,len(segments),BATCH_SIZE):
            batch_segments=segments[batch_start:batch_start+BATCH_SIZE]
            xb=torch.from_numpy(np.stack([to_logmel(item[0]) for item in batch_segments])).unsqueeze(1).to(DEVICE)
            with torch.no_grad(): probs=torch.sigmoid(model(xb)).cpu().numpy()
            for offset,((_,start_seconds,valid_seconds),prob) in enumerate(zip(batch_segments,probs)):
                limited_context=valid_seconds < DURATION_SECONDS
                predicted=[name for i,name in enumerate(class_order) if prob[i]>=REVIEW_THRESHOLD]
                review_score=max((float(prob[class_to_index[name]]) for name in REVIEW_CLASSES if name in class_to_index),default=0.0)
                alert_score=max((float(prob[class_to_index[name]]) for name in ALERT_CLASSES if name in class_to_index),default=0.0)
                rows.append({
                    'file':str(path.relative_to(PROJECT_ROOT)),
                    'segment_index':batch_start+offset,
                    'start_seconds':round(start_seconds,3),
                    'end_seconds':round(start_seconds+valid_seconds,3),
                    'valid_seconds':round(valid_seconds,3),
                    'status':'limited_context' if limited_context else 'predicted',
                    'predicted_labels':predicted,
                    'predicted_labels_text':('limited context: ' if limited_context else '')+(', '.join(predicted) if predicted else 'no class above review threshold'),
                    'max_score':float(prob.max()),
                    'review_score':review_score,
                    'alert_score':alert_score,
                    'review_recommended':limited_context or any(name in REVIEW_CLASSES and prob[j]>=REVIEW_THRESHOLD for j,name in enumerate(class_order)),
                    'high_priority_candidate':(not limited_context) and any(name in ALERT_CLASSES and prob[j]>=ALERT_THRESHOLD for j,name in enumerate(class_order)),
                    **{f'prob_{name}':float(prob[j]) for j,name in enumerate(class_order)},
                })
    except Exception as e:
        failures.append({'file':str(path),'error':f'{type(e).__name__}: {e}'})

predictions=pd.DataFrame(rows)
if predictions.empty:
    print('ไม่มี segment ที่ทำนายได้')
else:
    predictions=predictions.sort_values(['file','segment_index']).reset_index(drop=True)
    predictions_path=REPORT_DIR/'inference_predictions.csv'
    predictions.to_csv(predictions_path,index=False)
    summary=(predictions.groupby('file').agg(
        segments=('segment_index','count'),
        predicted_segments=('status',lambda s:int(s.eq('predicted').sum())),
        limited_context_segments=('status',lambda s:int(s.eq('limited_context').sum())),
        review_segments=('review_recommended','sum'),
        high_priority_segments=('high_priority_candidate','sum'),
        max_score=('max_score','max'),
        mean_score=('max_score','mean')
    ).reset_index())
    summary['detected_classes']=summary.file.map(lambda filename:', '.join(sorted({label for labels in predictions.loc[predictions.file.eq(filename)&predictions.status.eq('predicted'),'predicted_labels'] for label in labels})))
    summary['limited_context_candidates']=summary.file.map(lambda filename:', '.join(sorted({label for labels in predictions.loc[predictions.file.eq(filename)&predictions.status.eq('limited_context'),'predicted_labels'] for label in labels})))
    summary_path=REPORT_DIR/'inference_file_summary.csv'; summary.to_csv(summary_path,index=False)
    display(predictions[['file','start_seconds','end_seconds','valid_seconds','status','predicted_labels_text','review_score','alert_score']+[f'prob_{c}' for c in class_order]].head(30))
    display(summary.sort_values(['high_priority_segments','review_segments'],ascending=False).head(30))
    print('Final windows are aligned to the file end; only recordings shorter than one model window are padded and marked limited_context.')
    print('Saved:',predictions_path)
    print('Saved:',summary_path)
if failures:
    failures_path=REPORT_DIR/'inference_failures.csv'
    pd.DataFrame(failures).to_csv(failures_path,index=False)
    display(pd.DataFrame(failures))
    print('Saved:',failures_path)


Predict audio files: 100%|██████████| 2/2 [00:01<00:00,  1.40it/s]


,file,start_seconds,end_seconds,valid_seconds,status,predicted_labels_text,review_score,alert_score,prob_fire,prob_chainsaw,prob_vehicle,prob_human_activity,prob_threat_tactical,prob_weather,prob_natural_ambient,prob_silence,prob_other
0,data/incoming_audio/Car Passing By - Sound Eff...,0.000,4.000,4.0,predicted,"vehicle, other",0.803382,0.803382,0.008350,0.011620,0.803382,0.062905,0.043819,0.204373,0.051630,0.060411,0.602257
1,data/incoming_audio/Car Passing By - Sound Eff...,1.968,5.968,4.0,predicted,other,0.435097,0.435097,0.006431,0.010449,0.435097,0.312292,0.026676,0.117888,0.024189,0.001127,0.709582
2,data/incoming_audio/City Rain at Night Night T...,0.000,4.000,4.0,predicted,"weather, other",0.306136,0.306136,0.067915,0.039768,0.306136,0.183475,0.213152,0.721996,0.454518,0.427347,0.725209
3,data/incoming_audio/City Rain at Night Night T...,4.000,8.000,4.0,predicted,"weather, other",0.278804,0.278804,0.278804,0.054926,0.241257,0.187586,0.140102,0.720739,0.277976,0.139017,0.747646
4,data/incoming_audio/City Rain at Night Night T...,8.000,12.000,4.0,predicted,"weather, other",0.374712,0.374712,0.094512,0.046299,0.374712,0.129585,0.086578,0.526597,0.279593,0.146171,0.822304
5,data/incoming_audio/City Rain at Night Night T...,12.000,16.000,4.0,predicted,"vehicle, weather, other",0.698700,0.698700,0.064751,0.029846,0.698700,0.153930,0.107686,0.802952,0.160662,0.232864,0.521497
6,data/incoming_audio/City Rain at Night Night T...,16.000,20.000,4.0,predicted,"vehicle, weather, other",0.825296,0.825296,0.030774,0.015322,0.825296,0.091135,0.058750,0.631111,0.060344,0.124296,0.517802
7,data/incoming_audio/City Rain at Night Night T...,20.000,24.000,4.0,predicted,"vehicle, weather",0.767978,0.767978,0.046806,0.012005,0.767978,0.161850,0.089287,0.728776,0.094303,0.240110,0.493656
8,data/incoming_audio/City Rain at Night Night T...,24.000,28.000,4.0,predicted,"vehicle, weather",0.847654,0.847654,0.099509,0.018242,0.847654,0.084596,0.057001,0.655612,0.057256,0.071550,0.443417
9,data/incoming_audio/City Rain at Night Night T...,28.000,32.000,4.0,predicted,"vehicle, weather, other",0.857665,0.857665,0.131784,0.020950,0.857665,0.167208,0.106315,0.513066,0.085746,0.077883,0.503044


,file,segments,predicted_segments,limited_context_segments,review_segments,high_priority_segments,max_score,mean_score,detected_classes,limited_context_candidates
1,data/incoming_audio/City Rain at Night Night T...,23,23,0,7,4,0.893054,0.810186,"other, vehicle, weather",
0,data/incoming_audio/Car Passing By - Sound Eff...,2,2,0,1,1,0.803382,0.756482,"other, vehicle",


Final windows are aligned to the file end; only recordings shorter than one model window are padded and marked limited_context.
Saved: /Users/guidegiegg/Projects/RakForest-ML/reports/inference_predictions.csv
Saved: /Users/guidegiegg/Projects/RakForest-ML/reports/inference_file_summary.csv


## 7. ฟังตัวอย่างที่ควรตรวจ
เล่นช่วงที่ได้ review score สูงสุดไม่เกิน 10 ช่วง; แต่ละช่วงอาจมีหลาย label.

In [11]:
if not predictions.empty:
    preview=predictions[predictions.review_recommended].sort_values('review_score',ascending=False).head(10)
    for row in preview.itertuples():
        audio_path=PROJECT_ROOT/row.file
        start=int(row.start_seconds*SAMPLE_RATE)
        y,sr=read_audio_mono(audio_path)
        if sr!=SAMPLE_RATE: y=librosa.resample(y,orig_sr=sr,target_sr=SAMPLE_RATE)
        y=y[start:start+NUM_SAMPLES]
        display(Markdown(f'### {row.predicted_labels_text} | review score {row.review_score:.3f} | {row.file} @ {row.start_seconds:.1f}s'))
        display(Audio(y,rate=SAMPLE_RATE))
else:
    print('รันทำนายก่อน แล้วจึงแสดงตัวอย่างเสียงได้')


### vehicle, weather, other | review score 0.858 | data/incoming_audio/City Rain at Night Night Traffic Rain Video Rain Sounds for Slee (mp3cut.net).mp3 @ 28.0s

### vehicle, weather | review score 0.848 | data/incoming_audio/City Rain at Night Night Traffic Rain Video Rain Sounds for Slee (mp3cut.net).mp3 @ 24.0s

### vehicle, weather, other | review score 0.825 | data/incoming_audio/City Rain at Night Night Traffic Rain Video Rain Sounds for Slee (mp3cut.net).mp3 @ 16.0s

### vehicle, other | review score 0.803 | data/incoming_audio/Car Passing By - Sound Effect.mp3 @ 0.0s

### vehicle, weather | review score 0.768 | data/incoming_audio/City Rain at Night Night Traffic Rain Video Rain Sounds for Slee (mp3cut.net).mp3 @ 20.0s

### vehicle, weather, other | review score 0.699 | data/incoming_audio/City Rain at Night Night Traffic Rain Video Rain Sounds for Slee (mp3cut.net).mp3 @ 12.0s

### vehicle, weather, other | review score 0.536 | data/incoming_audio/City Rain at Night Night Traffic Rain Video Rain Sounds for Slee (mp3cut.net).mp3 @ 56.0s

### vehicle, weather, other | review score 0.527 | data/incoming_audio/City Rain at Night Night Traffic Rain Video Rain Sounds for Slee (mp3cut.net).mp3 @ 32.0s

## Outputs และขอบเขตการใช้งาน

- `reports/inference_predictions.csv`: ผลรายช่วง มีหลาย label และ score แยกคลาส; ช่วงของไฟล์ที่สั้นกว่า 4 วินาทีติดสถานะ `limited_context`
- `reports/inference_file_summary.csv`: คลาสที่พบ จำนวนช่วงที่ควรตรวจ และจำนวนช่วงที่มีบริบทสั้น
- `reports/inference_failures.csv`: ไฟล์ที่อ่านหรือทำนายไม่สำเร็จ (ถ้ามี)

Threshold 0.5/0.75 เป็นค่าเริ่มต้นที่ยังไม่ calibrate; `high_priority_candidate` ไม่ใช่คำสั่งแจ้งเหตุอัตโนมัติ ควรตรวจเสียงจริงและปรับเกณฑ์จาก validation/ข้อมูลภาคสนามก่อนใช้งานจริง. `silence` และ `other` เป็นแท็กเดี่ยว ไม่คาดว่าจะเกิดพร้อมคลาสอื่น.